# Track 1 Smart Campus - prediction notebook (c01_pattern_ridge)

Inference only: loads the trained artifact `model.pkl` (trained locally, see the training notebook) and predicts `energy_usage` for every row of the input file, in input order. All features are row-local, so shuffled rows / a missing `id` column are fine.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
# Feature engineering - identical to training (src/features.py)
import numpy as np
import pandas as pd

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A", "LIB_A",
             "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
TYPES = ["Administration", "Business", "Engineering", "LectureHall", "Library",
         "Residential", "Science", "Sports"]


def make_features(df):
    df = df.copy()
    out = pd.DataFrame(index=df.index)
    # fixed category lists -> identical integer codes at train and inference; unseen -> NaN
    out["building"] = pd.Categorical(df["building_id"], categories=BUILDINGS)
    out["btype"] = pd.Categorical(df["building_type"], categories=TYPES)
    out["hour"] = df["hour"].astype(float)
    out["dow"] = pd.Categorical(df["day_of_week"], categories=DAYS).codes.astype(float)
    out.loc[out["dow"] < 0, "dow"] = np.nan
    out["weekend"] = (out["dow"] >= 5).astype(float)
    out["month"] = df["month"].astype(float)
    for c in ["temperature", "humidity", "occupancy", "previous_usage"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    return out


NUMERIC = ["previous_usage", "occupancy", "temperature", "humidity"]


def linear_design(df, drop=()):
    """Design matrix for the per-building linear model (numpy, fixed column order).

    Per building: intercept, slopes on each observed numeric feature, 24 hour levels, weekend.
    Shared: annual sin/cos of month, centred temperature^2.
    `drop` lists numeric columns treated as unavailable (missingness-pattern sub-models).
    """
    n = len(df)
    B = np.stack([(df["building_id"].to_numpy() == b) for b in BUILDINGS], 1).astype(float)
    hour = df["hour"].to_numpy().astype(int)
    H = np.stack([(hour == h) for h in range(24)], 1).astype(float)
    we = df["day_of_week"].isin(["Saturday", "Sunday"]).to_numpy().astype(float)[:, None]
    m = df["month"].to_numpy().astype(float)
    cols = [B, (B[:, :, None] * H[:, None, :]).reshape(n, -1), B * we]
    for c in NUMERIC:
        if c not in drop:
            cols.append(B * df[c].to_numpy().astype(float)[:, None])
    cols.append(np.stack([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)], 1))
    if "temperature" not in drop:
        cols.append(((df["temperature"].to_numpy().astype(float) - 28.0) ** 2)[:, None])
    return np.concatenate(cols, 1)

In [ ]:
artifact = joblib.load("model.pkl")
test = pd.read_csv(TEST_PATH)
X_test = make_features(test)[artifact["features"]]
# one ridge per missingness pattern: each row uses the model fitted without its missing columns
miss = test[NUMERIC].isna().to_numpy()
keys = [tuple(sorted(c for c, mm in zip(NUMERIC, row) if mm)) for row in miss]
preds = np.empty(len(test))
for k in set(keys):
    idx = np.array([kk == k for kk in keys])
    preds[idx] = artifact["ridges"][k].predict(linear_design(test[idx], drop=set(k)))
preds = np.asarray(preds, dtype=float).ravel()
assert len(preds) == len(test) and np.isfinite(preds).all()

In [ ]:
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")